# Melbourne Suburb Analytics

In [2]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

projRoot = Path.cwd().parent
df = pd.read_csv(projRoot/"data"/"raw"/"melb_data.csv")

## Section 1: General Information & Cleansing
### 1.1 Initial Observations

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Address        13580 non-null  str    
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  str    
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  str    
 6   SellerG        13580 non-null  str    
 7   Date           13580 non-null  str    
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  str    
 17  Lattitude      13580 non-null  float64
 18  Longtitude     13

In [4]:
df["Car"].isna().sum()

np.int64(62)

In [5]:
df["BuildingArea"].isna().sum()

np.int64(6450)

In [6]:
df["YearBuilt"].isna().sum()

np.int64(5375)

In [7]:
df["CouncilArea"].isna().sum()

np.int64(1369)

In [8]:
df.drop(columns = ["Postcode"]).describe().style.format(lambda x: f'{x:6g}')

,Rooms,Price,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580,13580,13580,13580,13580,13518,13580,7130,8205,13580,13580,13580
mean,2.938,1.07568e+06,10.1378,2.91473,1.53424,1.61008,558.416,151.968,1964.68,-37.8092,144.995,7454.42
std,0.955748,639311,5.86872,0.965921,0.691712,0.962634,3990.67,541.015,37.2738,0.0792598,0.103916,4378.58
min,1,85000,0,0,0,0,0,0,1196,-38.1825,144.432,249
25%,2,650000,6.1,2,1,1,177,93,1940,-37.8568,144.93,4380
50%,3,903000,9.2,3,1,2,440,126,1970,-37.8024,145,6555
75%,3,1.33e+06,13,3,2,2,651,174,1999,-37.7564,145.058,10331
max,10,9e+06,48.1,20,8,10,433014,44515,2018,-37.4085,145.526,21650


<sub>◦ *"Postcode" column is dropped since it is categorical data;*</sub>  
<sub>◦ *style.format() is used to remove unnecessary decimals* </sub>

In [9]:
df.duplicated().sum()

np.int64(0)

**Missing values by column**
- `Car` : 62 missing
- `BuildingArea` : 6,450 missing
- `YearBuilt` : 5,375 missing
- `CouncilArea` : 1,369 missing

**Datatype errors**
- `Date` : string
- `Postcode` : float (categorical)

**Unusual Numbers**
- `BuildingArea` and `Landsize` shows a min of 0 & an absurd max size
- `Price` has a very large max

**Duplicated rows**
- There are `0` duplicated rows

### 1.2 Fixing data types

In [16]:
df["Date"] = pd.to_datetime(df["Date"], format = "%d/%m/%Y")

In [ ]:
df["Postcode"] = df["Postcode"].astype(str)

<StringDtype(storage='python', na_value=nan)>

**Conversions**  
<sub>  Filtering & Computation  </sub>
- `Date` : string -> datetime 

<sub>  Categorical </sub>
- `Postcode` : float -> string  

### 1.3 Resolving missing values

In [21]:
df.fillna({"Car": df["Car"].mean()}, inplace = True)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.79960,144.99840,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.80790,144.99340,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.80930,144.99440,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.79690,144.99690,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.80720,144.99410,Northern Metropolitan,4019.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,Wheelers Hill,12 Strada Cr,4,h,1245000.0,S,Barry,2017-08-26,16.7,3150.0,...,2.0,2.0,652.0,NaN,1981.0,NaN,-37.90562,145.16761,South-Eastern Metropolitan,7392.0
13576,Williamstown,77 Merrett Dr,3,h,1031000.0,SP,Williams,2017-08-26,6.8,3016.0,...,2.0,2.0,333.0,133.0,1995.0,NaN,-37.85927,144.87904,Western Metropolitan,6380.0
13577,Williamstown,83 Power St,3,h,1170000.0,S,Raine,2017-08-26,6.8,3016.0,...,2.0,4.0,436.0,NaN,1997.0,NaN,-37.85274,144.88738,Western Metropolitan,6380.0
13578,Williamstown,96 Verdon St,4,h,2500000.0,PI,Sweeney,2017-08-26,6.8,3016.0,...,1.0,5.0,866.0,157.0,1920.0,NaN,-37.85908,144.89299,Western Metropolitan,6380.0


In [23]:
df.fillna({"CouncilArea": "Unknown"}, inplace = True)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.79960,144.99840,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.80790,144.99340,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.80930,144.99440,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.79690,144.99690,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.80720,144.99410,Northern Metropolitan,4019.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,Wheelers Hill,12 Strada Cr,4,h,1245000.0,S,Barry,2017-08-26,16.7,3150.0,...,2.0,2.0,652.0,NaN,1981.0,Unknown,-37.90562,145.16761,South-Eastern Metropolitan,7392.0
13576,Williamstown,77 Merrett Dr,3,h,1031000.0,SP,Williams,2017-08-26,6.8,3016.0,...,2.0,2.0,333.0,133.0,1995.0,Unknown,-37.85927,144.87904,Western Metropolitan,6380.0
13577,Williamstown,83 Power St,3,h,1170000.0,S,Raine,2017-08-26,6.8,3016.0,...,2.0,4.0,436.0,NaN,1997.0,Unknown,-37.85274,144.88738,Western Metropolitan,6380.0
13578,Williamstown,96 Verdon St,4,h,2500000.0,PI,Sweeney,2017-08-26,6.8,3016.0,...,1.0,5.0,866.0,157.0,1920.0,Unknown,-37.85908,144.89299,Western Metropolitan,6380.0


**Outliers in data**
- Largest property price is **~9 million** while mean is only ~1.08 million
- Smallest land size is **0** which should not be possible
- Largest land size is **433014** which is massive considering the average is in the hundreds
- Smallest building area is **0** which should not be possible
- Largest building area is **44515** which is massive considering the average is in the hundreds